
# Qwen2.5-VL-7B-Instruct - Download & Test

This notebook downloads the Qwen2.5-VL-7B-Instruct vision-language model from Hugging Face
and tests it with sample identity document images from the KYC pipeline.

**Requirements:** ~16GB disk space for model download, ~8GB VRAM (GPU) recommended.
Runs on CPU if GPU unavailable (slower).



## Configuration

All model settings are in the cell below. Toggle flags to control behavior.


In [ ]:
# ============================================================
# QWEN2.5-VL CONFIGURATION (Everything in one cell)
# ============================================================

DOWNLOAD_MODEL = True   # Set to True to download model from Hugging Face
USE_QWEN_VL = True      # Set to True to use Qwen for extraction in tests

QWEN_VL_CONFIG = {
    "model_name": "Qwen/Qwen2.5-VL-7B-Instruct",
    "device": "auto",           # "auto", "cuda", "cpu"
    "torch_dtype": "bfloat16",  # "bfloat16", "float16", "float32"
    "max_tokens": 512,
    "temperature": 0.1,
    "force_download": False,     # Set True to re-download even if cached
    "local_cache_dir": None,     # Set to path like "./qwen_cache" or None (default)
}

# Sample images to test with
TEST_IMAGES = [
    "./sample_documents/pan_card.jpg",
    "./sample_documents/adhar_card.jpg",
    "./sample_documents/pic.jpg",
]

print("Configuration loaded")
print(f"  Download model: {DOWNLOAD_MODEL}")
print(f"  Use for extraction: {USE_QWEN_VL}")
print(f"  Model: {QWEN_VL_CONFIG['model_name']}")
print(f"  Device: {QWEN_VL_CONFIG['device']}")
print(f"  Test images: {len(TEST_IMAGES)}")

In [ ]:
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

import subprocess
import sys

packages = [
    "torch",
    "transformers",
    "accelerate",
    "qwen-vl-utils",
    "pillow",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All dependencies installed")

In [ ]:
# ============================================================
# IMPORTS (with conditional Qwen imports)
# ============================================================

import json
import re
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from PIL import Image
import time

# Qwen2.5-VL conditional imports
QWEN_VL_AVAILABLE = False
if USE_QWEN_VL or DOWNLOAD_MODEL:
    try:
        import torch
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        from qwen_vl_utils import process_vision_info
        QWEN_VL_AVAILABLE = True
        print("Qwen2.5-VL imports successful")
        print(f"  torch version: {torch.__version__}")
        print(f"  CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.get_device_name(0)}")
    except ImportError as e:
        print(f"Qwen2.5-VL imports failed: {e}")
        print("Run the install cell above first.")
        QWEN_VL_AVAILABLE = False

print("\nImports ready")


## Download Qwen2.5-VL-7B-Instruct

Downloads the model from Hugging Face hub (~15GB). Only runs if DOWNLOAD_MODEL=True.
If already cached, loads from cache.


In [ ]:
# ============================================================
# DOWNLOAD / LOAD QWEN2.5-VL MODEL
# ============================================================

if not QWEN_VL_AVAILABLE:
    raise RuntimeError("Qwen2.5-VL imports failed - cannot proceed")

qwen_model = None
qwen_processor = None

if DOWNLOAD_MODEL:
    model_name = QWEN_VL_CONFIG.get("model_name", "Qwen/Qwen2.5-VL-7B-Instruct")
    torch_dtype_str = QWEN_VL_CONFIG.get("torch_dtype", "bfloat16")
    torch_dtype = getattr(torch, torch_dtype_str, torch.bfloat16)
    device = QWEN_VL_CONFIG.get("device", "auto")
    force_dl = QWEN_VL_CONFIG.get("force_download", False)
    cache_dir = QWEN_VL_CONFIG.get("local_cache_dir", None)

    print(f"Loading model: {model_name}")
    print(f"  torch_dtype: {torch_dtype}")
    print(f"  device_map: {device}")
    if cache_dir:
        print(f"  cache_dir: {cache_dir}")
    print("  This may take several minutes...")
    start = time.time()

    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch_dtype,
        device_map=device,
        force_download=force_dl,
        cache_dir=cache_dir,
    )

    qwen_processor = AutoProcessor.from_pretrained(
        model_name,
        cache_dir=cache_dir,
    )

    elapsed = time.time() - start
    print(f"Model loaded in {elapsed:.1f}s")
else:
    print("DOWNLOAD_MODEL=False - skipping model download")
    print("Set DOWNLOAD_MODEL=True above and re-run to download")


## Extraction Functions

Defines Simulated OCR (fallback) and Qwen2.5-VL extraction.


In [ ]:
# ============================================================
# EXTRACTION FUNCTIONS
# ============================================================

def extract_simulated(file_path: str) -> Tuple[str, float]:
    """Simulated OCR extraction (fallback)"""
    try:
        img = Image.open(file_path)
        text = f"[Image Content from {Path(file_path).name}] "
        text += "Detected fields: ID Number: XYZ123456 | Expiry: 31-12-2030 | "
        text += "Issuer: Government Authority | Status: Valid"
        return text, 0.78
    except Exception as e:
        print(f"Simulated extraction error: {e}")
        return "", 0.0


def extract_qwen(file_path: str) -> Tuple[str, float]:
    """Extract text from image using Qwen2.5-VL"""
    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": file_path},
                    {"type": "text", "text": (
                        "Extract all visible text and structured information from this "
                        "identity document. Return each field on a new line as "
                        "key-value pairs, for example:\n"
                        "Name: John Doe\n"
                        "PAN: ABCDE1234F\n"
                        "DOB: 01-01-1990\n"
                        "Address: 123 Main St\n"
                        "ID: XYZ123456\n"
                        "Phone: +91-9876543210"
                    )},
                ],
            }
        ]

        text = qwen_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = qwen_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        if torch.cuda.is_available():
            inputs = inputs.to("cuda")

        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=QWEN_VL_CONFIG.get("max_tokens", 512),
            temperature=QWEN_VL_CONFIG.get("temperature", 0.1),
        )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = qwen_processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        result = output_text[0].strip() if output_text else ""
        text = f"[Image Content from {Path(file_path).name}] {result}"
        return text, 0.85

    except Exception as e:
        print(f"Qwen extraction error: {e}")
        return "", 0.0


def parse_fields(text: str) -> Dict[str, Any]:
    """Parse extracted text into structured fields"""
    fields = {}
    patterns = {
        "name": r"(?:Name|name)[:=]\s*([A-Za-z\s]+)",
        "pan": r"(?:PAN|pan)[:=]\s*([A-Z0-9]{10})",
        "dob": r"(?:DOB|dob|Date of Birth)[:=]\s*([0-9]{2}-[0-9]{2}-[0-9]{4})",
        "phone": r"(?:Phone|phone)[:=]\s*(\+?[0-9\-]+)",
        "address": r"(?:Address|address)[:=]\s*([A-Za-z0-9\s,]+)",
        "id_number": r"(?:ID|id)[:=]\s*([A-Z0-9]+)",
    }
    for field_name, pattern in patterns.items():
        match = re.search(pattern, text)
        if match:
            fields[field_name] = match.group(1).strip()
    return fields


def extract(file_path: str) -> Tuple[str, float]:
    """Dispatch: use Qwen if enabled and available, else simulated"""
    if USE_QWEN_VL and qwen_model is not None:
        return extract_qwen(file_path)
    return extract_simulated(file_path)


print("Extraction functions defined")


## Test Extraction on Sample Images

Runs extraction on each sample image using the selected method.


In [ ]:
# ============================================================
# TEST EXTRACTION ON SAMPLE IMAGES
# ============================================================

method = "Qwen2.5-VL" if (USE_QWEN_VL and qwen_model is not None) else "Simulated (Fallback)"
print(f"=" * 70)
print(f"  EXTRACTION METHOD: {method}")
print(f"=" * 70)

all_results = []

for img_path in TEST_IMAGES:
    path = Path(img_path)
    if not path.exists():
        print(f"\n  [!] File not found: {img_path}")
        continue

    print(f"\n  {'-' * 60}")
    print(f"  Document: {path.name}")
    print(f"  Path: {path.resolve()}")

    start = time.time()
    text, confidence = extract(img_path)
    elapsed = time.time() - start

    print(f"  Method: {method}")
    print(f"  Confidence: {confidence:.2f}")
    print(f"  Time: {elapsed:.2f}s")
    print(f"  Extracted text:")
    for line in text.split("|"):
        print(f"    {line.strip()}")

    fields = parse_fields(text)
    print(f"  Parsed fields: {fields if fields else '(none found)'}")

    all_results.append({
        "file": path.name,
        "confidence": confidence,
        "time_s": round(elapsed, 2),
        "text": text,
        "fields": fields,
    })

print(f"\n" + "=" * 70)
print(f"  EXTRACTION COMPLETE - {len(all_results)}/{len(TEST_IMAGES)} images processed")
print(f"=" * 70)


## Results Summary

Tabular comparison of all extraction results.


In [ ]:
# ============================================================
# RESULTS SUMMARY
# ============================================================

print(f"\n{'=' * 70}")
print(f"  RESULTS SUMMARY")
print(f"{'=' * 70}")
print(f"")
print(f"  {'File':<30} {'Confidence':<12} {'Time (s)':<10} {'Fields Found':<15}")
print(f"  {'-' * 30} {'-' * 12} {'-' * 10} {'-' * 15}")

total_conf = 0.0
total_time = 0.0

for r in all_results:
    field_count = len(r["fields"])
    print(f"  {r['file']:<30} {r['confidence']:<12.2f} {r['time_s']:<10.2f} {field_count:<15}")
    total_conf += r["confidence"]
    total_time += r["time_s"]

if all_results:
    avg_conf = total_conf / len(all_results)
    avg_time = total_time / len(all_results)
    print(f"  {'-' * 30} {'-' * 12} {'-' * 10} {'-' * 15}")
    print(f"  {'Average':<30} {avg_conf:<12.2f} {avg_time:<10.2f}")

print(f"\n  Method used: {method}")
print(f"  Model: {QWEN_VL_CONFIG['model_name'] if (qwen_model is not None) else 'N/A'}")


## Cleanup

Frees GPU memory by unloading the model.


In [ ]:
# ============================================================
# CLEANUP - Free GPU memory
# ============================================================

if qwen_model is not None:
    print("Cleaning up model from memory...")
    del qwen_model
    del qwen_processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    qwen_model = None
    qwen_processor = None
    print("Model unloaded, GPU memory freed")
else:
    print("No model to clean up")

print("\nDone")

In [ ]:
# ============================================================
# VERIFY: Can this model be used in the main KYC notebook?
# ============================================================

print("=" * 70)
print("  VERIFICATION: Integration with main KYC notebook")
print("=" * 70)

if qwen_model is not None:
    print("\n  Model is loaded and ready for use in the main pipeline.")
else:
    print("\n  Model not currently loaded (was cleaned up above).")

print("\n  To use this model in hackathon_improved.ipynb:")
print("    1. Open hackathon_improved.ipynb")
print("    2. In the CONFIGURATION cell, set:")
print(f'       USE_QWEN_VL = True')
print(f'       QWEN_VL_CONFIG["model_name"] = "{QWEN_VL_CONFIG["model_name"]}"')
print("    3. Re-run all cells (the model will load on first use)")

print("\n" + "=" * 70)
print("  NOTEBOOK COMPLETE")
print("=" * 70)